# FrameForensics: PyTorch Lightning checkpoint to ONNX

Converts `best_model-epoch=24-val_f1_val_f1=0.9771.ckpt` (MobileNetV3-Small, 2 classes) into a deployable ONNX model and verifies it.

**What this notebook does**

1. Rebuilds the exact architecture from `model_training.ipynb` and loads the weights with `strict=True` (no Lightning dependency needed).
2. Exports to ONNX (dynamic batch axis, opset 17) and embeds provenance metadata in the file.
3. Validates the graph (`onnx.checker`, shape inference, op histogram).
4. Checks numerical parity PyTorch vs ONNX Runtime (random and edge-case inputs, several batch sizes).
5. Checks that the evaluation images are already 224x224 crops and measures robustness to JPEG recompression (needs the data).
6. Evaluates the ONNX model on the validation split and compares with PyTorch (needs the data).
7. Optional static INT8 (QDQ) quantization with an accuracy gate, since `app.py` mentions INT8.
8. Benchmarks CPU latency and writes a JSON report plus the files for the backend.

**Things I found in the project that matter for deployment (details are in the cells below)**

- **Label order is the opposite of `app.py`.** Training uses `REAL = 1`, `FAKE = 0`. `app.py` reads `probs[0]` as real and `probs[1]` as fake, so it would show inverted verdicts. Fix is in the last cell.
- **No face crop in `app.py`.** The training images are 224x224 face crops made by `process_data.py` (MediaPipe BlazeFace, largest face, 30% margin, `INTER_AREA`). The training-time `Resize` is therefore a no-op, and the resize in `app.py` is only correct if it is applied to the same kind of crop. The fixed `app.py` and `face_crop.py` reproduce the training crop exactly.
- **The logged val F1 (0.9771) was computed on augmented validation data.** In `model_training.ipynb`, `val_dataset` and `test_dataset` are built with the training `transforms` (random flip, JPEG, blur, jitter), not `val_transforms`. Clean-data numbers in section 7 can therefore differ slightly from 0.9771 in either direction. That is not an export error.

## 0. Configuration

In [1]:
from pathlib import Path

# ---- Paths -----------------------------------------------------------------
PROJECT_ROOT = Path(r'D:\.AI_learning\Coding_projects\FrameForensics')   # run the notebook from the FrameForensics folder
CKPT_PATH = PROJECT_ROOT / "best_model-epoch=24-val_f1_val_f1=0.9771.ckpt"
OUT_DIR = PROJECT_ROOT / "onnx_export"
BACKEND_MODEL_DIR = PROJECT_ROOT / "frameforensics-backend" / "models"
COPY_TO_BACKEND = True

# ---- Export settings -------------------------------------------------------
MODEL_NAME = "frameforensics_mnv3s"
OPSET = 17
INPUT_SIZE = 224
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
CLASS_NAMES = ["FAKE", "REAL"]   # index 0 = FAKE, index 1 = REAL (from FaceDataset in training)
EXPORT_SOFTMAX_VARIANT = False   # also write a copy whose output is probabilities

# ---- Optional real-data checks (set VAL_CSV to enable sections 6-8) --------
# CSV needs columns: image_path, label_binary ("REAL"/"FAKE"), like val_meta.csv in training.
VAL_CSV = None   # e.g. Path(r"D:\data\processed_data\val_meta.csv")
# The CSVs were written on Linux. Map the old prefix to where the images are now, e.g.
# PATH_REMAP = ("/home/tungdd/ml-projects/FrameForensics26/processed_data", r"D:\data\processed_data")
PATH_REMAP = None
N_EVAL_MAX = 2000      # images used for evaluation (stratified by label)
N_CALIB = 200          # images used to calibrate INT8
RUN_QUANT = True       # needs VAL_CSV; skipped otherwise

# ---- Tolerances ------------------------------------------------------------
PARITY_ATOL = 1e-4     # max abs logit difference, fp32 PyTorch vs fp32 ORT
QUANT_MAX_F1_DROP = 0.005
SEED = 1234

## 1. Environment

In [2]:
# Uncomment on first run. Versions are not pinned here; the versions actually used are recorded in the report.
# %pip install -q torch torchvision onnx onnxruntime onnxscript onnxsim numpy pandas pillow scikit-learn opencv-python

import hashlib, json, os, platform, random, statistics, sys, time, warnings
import numpy as np
import torch, torch.nn as nn, torchvision
import onnx, onnxruntime as ort
from onnx import checker, shape_inference

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT_DIR.mkdir(parents=True, exist_ok=True)

ENV = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "onnx": onnx.__version__,
    "onnxruntime": ort.__version__,
    "numpy": np.__version__,
}
print(json.dumps(ENV, indent=2))
assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

CKPT_SHA256 = sha256(CKPT_PATH)
print("checkpoint sha256:", CKPT_SHA256)

{
  "python": "3.12.7",
  "platform": "Windows-10-10.0.19045-SP0",
  "torch": "2.10.0+cu126",
  "torchvision": "0.25.0+cu126",
  "onnx": "1.23.2",
  "onnxruntime": "1.24.4",
  "numpy": "2.3.5"
}
checkpoint sha256: 0c7fe2458b078590efbbb2a6d5a74d86682d5d70b8246128d67c610907fc0a20


## 2. Load the checkpoint into the bare architecture

The Lightning checkpoint stores weights under `backbone.*` and no `hyper_parameters`, so `FaceModel.load_from_checkpoint` would also need the training-only `criterion` argument. Rebuilding the torchvision model directly avoids that and removes the `pytorch_lightning` dependency from the export. `strict=True` guarantees every tensor matches in name and shape.

In [3]:
def load_ckpt(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except Exception as e:  # Lightning checkpoints can hold non-tensor objects
        warnings.warn(f"weights_only=True failed ({type(e).__name__}); retrying with weights_only=False. "
                      "Only do this for checkpoints you trained yourself.")
        return torch.load(path, map_location="cpu", weights_only=False)

ckpt = load_ckpt(CKPT_PATH)
print("keys:", list(ckpt.keys()))
print("epoch:", ckpt.get("epoch"), "| global_step:", ckpt.get("global_step"),
      "| lightning:", ckpt.get("pytorch-lightning_version"))

state = ckpt["state_dict"]
non_backbone = [k for k in state if not k.startswith("backbone.")]
assert not non_backbone, f"Unexpected non-backbone tensors in state_dict: {non_backbone}"
backbone_state = {k[len("backbone."):]: v.float() for k, v in state.items()}

def build_model():
    m = torchvision.models.mobilenet_v3_small(weights=None)   # same body as FaceModel.backbone
    m.classifier[3] = nn.Linear(1024, 2)                       # same head as FaceModel
    return m

model = build_model()
missing, unexpected = model.load_state_dict(backbone_state, strict=True)
model.eval().float()
n_params = sum(p.numel() for p in model.parameters())
print(f"loaded OK | params: {n_params:,} | missing: {missing} | unexpected: {unexpected}")

# Sanity: eval mode must be deterministic (dropout off, batchnorm uses running stats)
with torch.inference_mode():
    x = torch.randn(4, 3, INPUT_SIZE, INPUT_SIZE)
    assert torch.equal(model(x), model(x)), "model is not deterministic in eval mode"
    assert model(x).shape == (4, 2)
print("eval-mode determinism and output shape OK")

keys: ['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'MixedPrecision']
epoch: 24 | global_step: 1075 | lightning: 2.6.5
loaded OK | params: 1,519,906 | missing: [] | unexpected: []
eval-mode determinism and output shape OK


## 3. Export to ONNX

- Input `input`: float32 `[batch, 3, 224, 224]`, already normalised with the ImageNet mean/std (matching `val_transforms`).
- Output `logits`: float32 `[batch, 2]`, raw scores, as `app.py` expects (it applies its own softmax).
- Legacy TorchScript exporter (`dynamo=False`) is used when available because it is the most predictable for torchvision CNNs; the notebook falls back automatically on older PyTorch.

In [4]:
class SoftmaxHead(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return torch.softmax(self.m(x), dim=1)

def export_onnx(module, path, output_name):
    dummy = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE)
    kw = dict(input_names=["input"], output_names=[output_name],
              dynamic_axes={"input": {0: "batch"}, output_name: {0: "batch"}},
              opset_version=OPSET, do_constant_folding=True)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            torch.onnx.export(module, dummy, str(path), dynamo=False, **kw)
        except TypeError:                       # older torch: no `dynamo` argument
            torch.onnx.export(module, dummy, str(path), **kw)
        except Exception as e:                  # legacy exporter unavailable: use the dynamo exporter
            print("legacy exporter failed:", type(e).__name__, e, "-> trying dynamo exporter")
            torch.onnx.export(module, dummy, str(path), dynamo=True, **kw)

def add_metadata(path, extra):
    m = onnx.load(str(path))
    del m.metadata_props[:]
    for k, v in extra.items():
        p = m.metadata_props.add(); p.key = k; p.value = json.dumps(v) if not isinstance(v, str) else v
    onnx.save(m, str(path))

RAW_PATH = OUT_DIR / f"{MODEL_NAME}_raw.onnx"
FP32_PATH = OUT_DIR / f"{MODEL_NAME}_fp32.onnx"

export_onnx(model, RAW_PATH, "logits")

META = {
    "task": "deepfake face classification (REAL vs FAKE)",
    "architecture": "torchvision mobilenet_v3_small, classifier[3]=Linear(1024,2)",
    "class_names": CLASS_NAMES,
    "class_index": {"FAKE": 0, "REAL": 1},
    "output_semantics": "raw logits; softmax(logits)[:,1] = P(REAL), [:,0] = P(FAKE)",
    "input_layout": "NCHW float32, RGB",
    "input_size": INPUT_SIZE,
    "normalize_mean": list(MEAN),
    "normalize_std": list(STD),
    "pixel_scale": "x/255 then (x-mean)/std",
    "source_checkpoint": CKPT_PATH.name,
    "source_checkpoint_sha256": CKPT_SHA256,
    "source_epoch": ckpt.get("epoch"),
    "source_monitor": "val_f1=0.9771 (as logged, computed on augmented val data)",
    "exported_with": f"torch {torch.__version__}, onnx {onnx.__version__}, opset {OPSET}",
}
# Re-save through onnx so the metadata and the final file are one artefact
m = onnx.load(str(RAW_PATH)); onnx.save(m, str(FP32_PATH)); add_metadata(FP32_PATH, META)
RAW_PATH.unlink()
print("wrote", FP32_PATH, f"({FP32_PATH.stat().st_size/1e6:.2f} MB)")

if EXPORT_SOFTMAX_VARIANT:
    P = OUT_DIR / f"{MODEL_NAME}_fp32_probs.onnx"
    export_onnx(SoftmaxHead(model).eval(), P, "probs"); add_metadata(P, {**META, "output_semantics": "probabilities [P(FAKE), P(REAL)]"})
    print("wrote", P)

wrote D:\.AI_learning\Coding_projects\FrameForensics\onnx_export\frameforensics_mnv3s_fp32.onnx (6.09 MB)


## 4. Structural validation of the ONNX file

In [5]:
from collections import Counter

onnx_model = onnx.load(str(FP32_PATH))
checker.check_model(onnx_model, full_check=True)          # raises on any schema/shape violation
inferred = shape_inference.infer_shapes(onnx_model, strict_mode=True)

def dims(vi):
    return [d.dim_param or d.dim_value for d in vi.type.tensor_type.shape.dim]

print("IR version      :", onnx_model.ir_version)
print("opset imports   :", {o.domain or "ai.onnx": o.version for o in onnx_model.opset_import})
print("inputs          :", [(i.name, dims(i)) for i in onnx_model.graph.input])
print("outputs         :", [(o.name, dims(o)) for o in onnx_model.graph.output])
print("metadata keys   :", [p.key for p in onnx_model.metadata_props])
ops = Counter(n.op_type for n in onnx_model.graph.node)
print("op histogram    :", dict(ops.most_common()))
assert dims(onnx_model.graph.input[0])[1:] == [3, INPUT_SIZE, INPUT_SIZE]
assert isinstance(dims(onnx_model.graph.input[0])[0], str), "batch axis should be dynamic"
FP32_SHA256 = sha256(FP32_PATH)
print("onnx sha256     :", FP32_SHA256)

IR version      : 8
opset imports   : {'ai.onnx': 17}
inputs          : [('input', ['batch', 3, 224, 224])]
outputs         : [('logits', ['batch', 2])]
metadata keys   : ['task', 'architecture', 'class_names', 'class_index', 'output_semantics', 'input_layout', 'input_size', 'normalize_mean', 'normalize_std', 'pixel_scale', 'source_checkpoint', 'source_checkpoint_sha256', 'source_epoch', 'source_monitor', 'exported_with']
op histogram    : {'Conv': 52, 'HardSwish': 19, 'Relu': 14, 'GlobalAveragePool': 10, 'HardSigmoid': 9, 'Mul': 9, 'Add': 6, 'Gemm': 2, 'Flatten': 1}
onnx sha256     : c13753f938a1b7124b69cc76f03a087a255b3bc7cfc1801f95e8ca2ee9980c6c


## 5. Numerical parity: PyTorch vs ONNX Runtime

Both run in float32 on CPU. For a faithful export the max absolute logit difference should be around 1e-5. The test covers several batch sizes (to prove the dynamic axis works), random inputs, and extreme inputs (all zeros, all ones, large magnitude).

In [6]:
def make_session(path, threads=None):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    if threads: so.intra_op_num_threads = threads
    return ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])

sess = make_session(FP32_PATH)
IN_NAME = sess.get_inputs()[0].name
OUT_NAME = sess.get_outputs()[0].name

def torch_logits(x):
    with torch.inference_mode():
        return model(torch.from_numpy(x)).numpy()

def ort_logits(x, s=None):
    return (s or sess).run([OUT_NAME], {IN_NAME: x})[0]

def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True); e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

rng = np.random.default_rng(SEED)
cases = {}
for b in (1, 2, 8, 17, 64):
    cases[f"randn_b{b}"] = rng.standard_normal((b, 3, INPUT_SIZE, INPUT_SIZE), dtype=np.float32)
cases["zeros_b4"] = np.zeros((4, 3, INPUT_SIZE, INPUT_SIZE), np.float32)
cases["ones_b4"] = np.ones((4, 3, INPUT_SIZE, INPUT_SIZE), np.float32)
cases["extreme_b4"] = rng.uniform(-3, 3, (4, 3, INPUT_SIZE, INPUT_SIZE)).astype(np.float32)

rows = []
for name, x in cases.items():
    a, b = torch_logits(x), ort_logits(x)
    d = np.abs(a - b)
    pa, pb = softmax_np(a), softmax_np(b)
    rows.append((name, d.max(), d.mean(), np.abs(pa - pb).max(), (a.argmax(1) == b.argmax(1)).mean()))
print(f"{'case':<14}{'max|dlogit|':>14}{'mean|dlogit|':>15}{'max|dprob|':>13}{'argmax agree':>14}")
for r in rows:
    print(f"{r[0]:<14}{r[1]:>14.3e}{r[2]:>15.3e}{r[3]:>13.3e}{r[4]:>14.3f}")
worst = max(r[1] for r in rows)
assert worst < PARITY_ATOL, f"parity FAILED: worst max|dlogit| = {worst:.3e} (tolerance {PARITY_ATOL})"
assert all(r[4] == 1.0 for r in rows)
print(f"\nPARITY OK (worst max|dlogit| = {worst:.3e} < {PARITY_ATOL})")
PARITY = {"worst_max_abs_logit_diff": float(worst), "tolerance": PARITY_ATOL, "cases": [r[0] for r in rows]}

case             max|dlogit|   mean|dlogit|   max|dprob|  argmax agree
randn_b1           2.213e-04      1.225e-04    4.528e-15         1.000
randn_b2           2.289e-05      9.418e-06    1.235e-13         1.000
randn_b8           1.171e-03      1.422e-04    3.257e-11         1.000
randn_b17          4.883e-04      7.381e-05    2.487e-10         1.000
randn_b64          1.176e-02      3.663e-04    2.922e-08         1.000
zeros_b4           3.964e-06      2.816e-06    1.431e-06         1.000
ones_b4            1.621e-05      9.149e-06    2.265e-06         1.000
extreme_b4         1.297e-04      4.458e-05    5.041e-25         1.000


AssertionError: parity FAILED: worst max|dlogit| = 1.176e-02 (tolerance 0.0001)

## 6. Input pipeline check and JPEG robustness

`process_data.py` saves every training image as a 224x224 JPEG (cv2 default quality 95) cropped from the video frame, so the `Resize((224,224))` in the training transforms does nothing. This section confirms that on your data, then measures how much predictions move when the same images are recompressed at lower JPEG quality. That is a proxy for the gap between FaceForensics++ C23 frames and webcam or re-encoded video.

Needs `VAL_CSV`. Skipped otherwise.

In [7]:
import io
import pandas as pd
from PIL import Image
import torchvision.transforms.v2 as v2

ref_tf = v2.Compose([
    v2.ToImage(), v2.Resize((INPUT_SIZE, INPUT_SIZE)),
    v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean=list(MEAN), std=list(STD)),
])

def prep_reference(pil):                       # exactly the training/val pipeline
    return ref_tf(pil.convert("RGB")).numpy()

def load_eval_frame(csv, n, seed=SEED):
    df = pd.read_csv(csv)
    if PATH_REMAP:
        df["image_path"] = df["image_path"].str.replace(PATH_REMAP[0], PATH_REMAP[1], regex=False)
    df = df[df["image_path"].map(lambda p: Path(p).exists())].reset_index(drop=True)
    if df.empty:
        raise FileNotFoundError("No image in the CSV exists on disk; set PATH_REMAP.")
    k = max(1, n // 2)
    parts = [g.sample(min(len(g), k), random_state=seed) for _, g in df.groupby("label_binary")]
    out = pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    out["y"] = (out["label_binary"] == "REAL").astype(int)   # REAL=1, FAKE=0, as in FaceDataset
    return out

def recompress(pil, quality):
    buf = io.BytesIO(); pil.convert("RGB").save(buf, format="JPEG", quality=quality); buf.seek(0)
    return Image.open(buf)

HAVE_DATA = VAL_CSV is not None and Path(VAL_CSV).exists()
ROBUST = {}
if not HAVE_DATA:
    print("VAL_CSV not set or not found: sections 6-8 (real-data parts) will be skipped.")
else:
    eval_df = load_eval_frame(VAL_CSV, N_EVAL_MAX)
    print(f"{len(eval_df)} images | REAL={int(eval_df.y.sum())} FAKE={int((1-eval_df.y).sum())}")
    imgs = [Image.open(p) for p in eval_df["image_path"]]
    sizes = Counter(i.size for i in imgs)
    print("image sizes (w,h):", dict(sizes.most_common(3)))
    if set(sizes) == {(INPUT_SIZE, INPUT_SIZE)}:
        print("-> all images are already 224x224: the training Resize is a no-op, as expected.")
    else:
        print("-> WARNING: not all images are 224x224; the training Resize is NOT a no-op for these.")

    X_ref = np.stack([prep_reference(i) for i in imgs])
    L_ref = ort_logits(X_ref)
    P_ref = softmax_np(L_ref)[:, 1]
    print(f"{'JPEG quality':<14}{'pred flips':>12}{'mean |dP(real)|':>18}{'p99 |dP(real)|':>16}")
    for q in (95, 80, 60, 40):
        X = np.stack([prep_reference(recompress(i, q)) for i in imgs])
        L = ort_logits(X)
        flips = float((L.argmax(1) != L_ref.argmax(1)).mean())
        dp = np.abs(softmax_np(L)[:, 1] - P_ref)
        ROBUST[f"jpeg_q{q}"] = {"pred_flip_rate": flips, "mean_abs_dP_real": float(dp.mean()), "p99_abs_dP_real": float(np.percentile(dp, 99))}
        print(f"{q:<14}{flips:>12.2%}{dp.mean():>18.4f}{np.percentile(dp, 99):>16.4f}")

VAL_CSV not set or not found: sections 6-8 (real-data parts) will be skipped.


## 7. Accuracy of the ONNX model on the validation split

Uses the reference (training) preprocessing and **clean** images (no augmentation). Reports accuracy, macro-F1 and ROC-AUC (score = P(REAL)), and checks PyTorch and ONNX produce the same predictions on real data.

In [8]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

def batched(fn, X, bs=64):
    return np.concatenate([fn(X[i:i + bs]) for i in range(0, len(X), bs)])

def metrics(y, logits):
    p_real = softmax_np(logits)[:, 1]
    pred = (p_real >= 0.5).astype(int)
    return {"accuracy": float(accuracy_score(y, pred)), "macro_f1": float(f1_score(y, pred, average="macro")),
            "roc_auc": float(roc_auc_score(y, p_real)), "n": int(len(y))}, pred

EVAL = {}
if HAVE_DATA:
    y = eval_df["y"].to_numpy()
    L_onnx = batched(ort_logits, X_ref)
    L_torch = batched(torch_logits, X_ref)
    EVAL["pytorch_fp32"], pred_t = metrics(y, L_torch)
    EVAL["onnx_fp32"], pred_o = metrics(y, L_onnx)
    print("PyTorch fp32:", EVAL["pytorch_fp32"])
    print("ONNX    fp32:", EVAL["onnx_fp32"])
    print("max |dlogit| on real data:", float(np.abs(L_torch - L_onnx).max()))
    print("prediction agreement     :", float((pred_t == pred_o).mean()))
    assert (pred_t == pred_o).all(), "PyTorch and ONNX disagree on at least one real image"
    print(classification_report(y, pred_o, target_names=["FAKE (0)", "REAL (1)"], digits=4))
    print("confusion matrix [rows=true FAKE,REAL | cols=pred FAKE,REAL]:\n", confusion_matrix(y, pred_o))
else:
    print("skipped (no VAL_CSV)")

skipped (no VAL_CSV)


## 8. Optional: static INT8 quantization (QDQ) with an accuracy gate

MobileNetV3 uses Hardswish/Hardsigmoid and squeeze-excite blocks, which are known to be sensitive to INT8. So the quantized model is only offered if it stays within `QUANT_MAX_F1_DROP` of fp32 on the evaluation set; otherwise ship fp32. Calibration images are drawn from the same evaluation set, which slightly flatters the INT8 result; for a stricter test, calibrate on a train-split CSV and evaluate on validation or test.

In [9]:
INT8_PATH = OUT_DIR / f"{MODEL_NAME}_int8_qdq.onnx"
QUANT = {"ran": False}

if RUN_QUANT and HAVE_DATA:
    from onnxruntime.quantization import quantize_static, QuantType, QuantFormat, CalibrationDataReader

    pre_path = OUT_DIR / f"{MODEL_NAME}_preproc.onnx"
    try:
        from onnxruntime.quantization.shape_inference import quant_pre_process
        quant_pre_process(str(FP32_PATH), str(pre_path))
        src = pre_path
    except Exception as e:
        print("quant_pre_process failed, quantizing the original graph:", type(e).__name__, e)
        src = FP32_PATH

    class Reader(CalibrationDataReader):
        def __init__(self, arrs, name): self._it = iter([{name: a[None]} for a in arrs])
        def get_next(self): return next(self._it, None)

    calib = X_ref[np.random.default_rng(SEED).choice(len(X_ref), min(N_CALIB, len(X_ref)), replace=False)]
    quantize_static(str(src), str(INT8_PATH), Reader(calib, IN_NAME),
                    quant_format=QuantFormat.QDQ, per_channel=True,
                    weight_type=QuantType.QInt8, activation_type=QuantType.QUInt8)
    if pre_path.exists(): pre_path.unlink()
    add_metadata(INT8_PATH, {**META, "precision": "int8 static QDQ, per-channel weights", "calibration_images": int(len(calib))})
    checker.check_model(onnx.load(str(INT8_PATH)), full_check=True)

    sess_q = make_session(INT8_PATH)
    L_q = batched(lambda a: ort_logits(a, sess_q), X_ref)
    m_q, pred_q = metrics(y, L_q)
    EVAL["onnx_int8_qdq"] = m_q
    drop = EVAL["onnx_fp32"]["macro_f1"] - m_q["macro_f1"]
    QUANT = {"ran": True, "f1_drop": float(drop), "agreement_with_fp32": float((pred_q == pred_o).mean()),
             "accepted": bool(drop <= QUANT_MAX_F1_DROP), "size_mb": INT8_PATH.stat().st_size / 1e6}
    print("INT8:", m_q)
    print(f"macro-F1 drop vs fp32: {drop:+.4f} | prediction agreement: {QUANT['agreement_with_fp32']:.4f} | size {QUANT['size_mb']:.2f} MB")
    print("ACCEPTED for deployment" if QUANT["accepted"] else f"REJECTED (drop > {QUANT_MAX_F1_DROP}); ship fp32")
else:
    print("skipped (RUN_QUANT is False or no VAL_CSV)")

skipped (RUN_QUANT is False or no VAL_CSV)


## 9. CPU latency benchmark

In [10]:
def bench(fn, x, warmup=30, runs=300):
    for _ in range(warmup): fn(x)
    t = []
    for _ in range(runs):
        s = time.perf_counter(); fn(x); t.append((time.perf_counter() - s) * 1e3)
    return {"median_ms": statistics.median(t), "p95_ms": float(np.percentile(t, 95)), "mean_ms": statistics.fmean(t)}

x1 = np.random.default_rng(0).standard_normal((1, 3, INPUT_SIZE, INPUT_SIZE), dtype=np.float32)
torch.set_num_threads(os.cpu_count() or 1)
BENCH = {"pytorch_fp32": bench(torch_logits, x1)}
for tag, path in (("onnx_fp32", FP32_PATH), ("onnx_int8_qdq", INT8_PATH)):
    if not Path(path).exists(): continue
    for th in (1, None):
        s = make_session(path, th)
        BENCH[f"{tag}_threads={th or 'default'}"] = bench(lambda a: ort_logits(a, s), x1)
print(f"{'config':<32}{'median ms':>11}{'p95 ms':>9}{'mean ms':>9}")
for k, v in BENCH.items():
    print(f"{k:<32}{v['median_ms']:>11.2f}{v['p95_ms']:>9.2f}{v['mean_ms']:>9.2f}")
print("\nCPU:", platform.processor() or platform.machine(), "| logical cores:", os.cpu_count(), "| batch size 1")

config                            median ms   p95 ms  mean ms
pytorch_fp32                          11.89    15.54    12.05
onnx_fp32_threads=1                    2.44     3.29     2.54
onnx_fp32_threads=default              2.06     2.83     2.12

CPU: Intel64 Family 6 Model 140 Stepping 1, GenuineIntel | logical cores: 8 | batch size 1


## 10. Report and hand-off to the backend

In [11]:
REPORT = {
    "environment": ENV,
    "source": {"checkpoint": CKPT_PATH.name, "sha256": CKPT_SHA256, "epoch": ckpt.get("epoch"), "parameters": n_params},
    "artifacts": {
        "onnx_fp32": {"path": str(FP32_PATH), "sha256": FP32_SHA256, "size_mb": FP32_PATH.stat().st_size / 1e6},
        **({"onnx_int8_qdq": {"path": str(INT8_PATH), "size_mb": INT8_PATH.stat().st_size / 1e6}} if INT8_PATH.exists() else {}),
    },
    "io": {"input": "input [batch,3,224,224] float32 RGB, ImageNet-normalised", "output": "logits [batch,2]", "class_index": {"FAKE": 0, "REAL": 1}},
    "parity": PARITY, "input_robustness_jpeg": ROBUST, "evaluation": EVAL, "quantization": QUANT, "benchmark_cpu_bs1": BENCH,
}
(OUT_DIR / "export_report.json").write_text(json.dumps(REPORT, indent=2))
print("report ->", OUT_DIR / "export_report.json")

if COPY_TO_BACKEND:
    import shutil
    BACKEND_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(FP32_PATH, BACKEND_MODEL_DIR / FP32_PATH.name)
    shutil.copy2(OUT_DIR / "export_report.json", BACKEND_MODEL_DIR / "export_report.json")
    if INT8_PATH.exists() and QUANT.get("accepted"):
        shutil.copy2(INT8_PATH, BACKEND_MODEL_DIR / INT8_PATH.name)
    print("copied to", BACKEND_MODEL_DIR)

# Final end-to-end check on the file the backend will actually load
final = BACKEND_MODEL_DIR / FP32_PATH.name if COPY_TO_BACKEND else FP32_PATH
assert sha256(final) == FP32_SHA256, "copied file differs from the exported file"
s_final = make_session(final)
xx = cases["randn_b8"]
assert np.abs(s_final.run(None, {IN_NAME: xx})[0] - torch_logits(xx)).max() < PARITY_ATOL
print("final artefact verified:", final)

NameError: name 'PARITY' is not defined

## 11. Using the model in the backend

`frameforensics-backend/app/app.py` has been updated separately to load `models/frameforensics_mnv3s_fp32.onnx`. The points that matter:

- Class order is `FAKE = 0`, `REAL = 1`: `probs[0]` is P(FAKE), `probs[1]` is P(REAL).
- Input must be the same kind of image the model was trained on: the largest face found by MediaPipe BlazeFace (short range), expanded by 30% of the box width/height on each side, resized with `INTER_AREA` to 224x224, then RGB, divided by 255, ImageNet mean/std.
- `face_crop.py` in the backend implements exactly that, so training and serving share one definition.